# ЛР 13. Шаблонный ретросинтез и выбор маршрута

**Сущность:** Выполнить прозрачное структурное разбиение и проверить упрощённую экономическую модель.

**Данные:** Цель — метилацетат COC(C)=O; routes.csv содержит два вымышленных линейных маршрута.

**Workflow:** Эфир → реакционный SMARTS → санитарная проверка продуктов → выходы → сравнение допущений.

Перед запуском прочитайте [guideline](../guides/lab13.md) и [теорию](../../Module_theory/Module%2013_retrosynthesis.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [1]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab13"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Найдите сложноэфирный фрагмент и примените один заданный шаблон.
2. Проверьте, что получены кислота и спирт; это ещё не препаративный протокол.
3. Сравните два маршрута по произведению долей и заданным суммарным затратам.

Ожидаемое: Продукты метилацетата: уксусная кислота и метанол; выходы A=0,56/B=0,855; приведённые затраты A≈1071,43/B≈877,19 при явно заданной модели.

In [2]:
from rdkit import Chem
from rdkit.Chem import rdChemReactions
reaction=rdChemReactions.ReactionFromSmarts("[C:1](=[O:2])[O:3][C:4]>>[C:1](=[O:2])[OH:3].[OH][C:4]")
products=reaction.RunReactants((Chem.MolFromSmiles("COC(C)=O"),))
assert products
names=[]
for mol in products[0]:
    Chem.SanitizeMol(mol)
    for atom in mol.GetAtoms(): atom.SetAtomMapNum(0)
    names.append(Chem.MolToSmiles(mol))
print("Учебные предшественники:",names)
assert set(names)=={"CC(=O)O","CO"}
routes=pd.read_csv(DATA/"routes.csv")
routes["overall_yield"]=routes.yield1*routes.yield2
routes["normalized_cost"]=routes.assumed_cost/routes.overall_yield
display(routes)
assert np.allclose(routes.overall_yield,[0.56,0.855])
assert routes.loc[1,"normalized_cost"]<routes.loc[0,"normalized_cost"]
routes.to_csv(OUT/"route_comparison.csv",index=False)
artifact={"precursors":names,"routes":routes.to_dict(orient="records"),"scope":"one ester template, not general AI retrosynthesis"}


Учебные предшественники: ['CC(=O)O', 'CO']


,route,yield1,yield2,assumed_cost,overall_yield,normalized_cost
0,A,0.80,0.7,600,0.560,1071.428571
1,B,0.95,0.9,750,0.855,877.192982


## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [3]:
elapsed = time.perf_counter()-started
run_record = {"lab": 13, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\Labs\results\lab13\benchmark.json


## 4. Самостоятельная тренировка

1. Примените шаблон к этилацетату; определите второй продукт.
2. Назовите три причины, по которым реальный выход может отличаться.
3. Почему SA Score не заменяет маршрут и прайс-лист?

[Ответы после попытки](../solutions/lab13.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.